# Cell 1：安裝套件

In [25]:
!pip install google-generativeai python-dotenv

# Cell 2：設定 API Key 與路徑

In [ ]:
import os

# Gemini API Key
os.environ["GEMINI_API_KEY"] = "GEMINI_API_KEY"

# 路徑設定
INPUT_PATH  = "clustered_events.json"


OUTPUT_DIR  = "phase3_output_07"   # 報告輸出資料夾（建立）

### 設定 `INPUT_PATH` (使用 `os.listdir()`)
搜尋符合 `clustered_events*.json` 模式的檔案。

In [27]:
# 取得 INPUT_PATH 所在的目錄，如果 INPUT_PATH 只是檔名，則預設為當前目錄
search_dir = os.path.dirname(INPUT_PATH) if os.path.dirname(INPUT_PATH) else "."

found_file_path = None
for filename in os.listdir(search_dir):
    if filename.startswith("clustered_events") and filename.endswith(".json") and "clustered_events" in filename:
        found_file_path = os.path.join(search_dir, filename)
        break

if found_file_path:
    INPUT_PATH = found_file_path
    print(f"已動態設定 INPUT_PATH 為：{INPUT_PATH}")
else:
    print(f"[WARNING] 未找到符合 'clustered_events*.json' 模式的檔案，請確認檔案是否存在。目前 INPUT_PATH 為：{INPUT_PATH}")


已動態設定 INPUT_PATH 為：./clustered_events.json


# Cell 3：Import 與初始化

In [28]:
import json
import google.generativeai as genai
import requests
import time
import os

api_key = os.environ.get("GEMINI_API_KEY")
genai.configure(api_key=api_key)
print(f"API Key 設定完成")
print(f"輸入路徑：{INPUT_PATH}")
print(f"輸出資料夾：{OUTPUT_DIR}")

API Key 設定完成
輸入路徑：./clustered_events.json
輸出資料夾：phase3_output_07


測試api狀態

In [29]:
'''
import requests, os, json

url = "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent"
params = {"key": os.environ.get("GEMINI_API_KEY")}
body = {
    "contents": [{"parts": [{"text": "Say hello in one word"}]}]
}

resp = requests.post(url, params=params, json=body, timeout=30)
print(resp.status_code)
print(resp.text[:300])
'''

'\nimport requests, os, json\n\nurl = "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent"\nparams = {"key": os.environ.get("GEMINI_API_KEY")}\nbody = {\n    "contents": [{"parts": [{"text": "Say hello in one word"}]}]\n}\n\nresp = requests.post(url, params=params, json=body, timeout=30)\nprint(resp.status_code)\nprint(resp.text[:300])\n'

# Cell 4：定義所有函式
4-1：載入 clustered_events.json

In [30]:
def load_clusters(input_path: str) -> list:
    """
    讀取 Phase 2 輸出的 clustered_events.json
    因為 Phase 2 config 設定 include_noise=false，
    所以檔案裡不會有 noise cluster，全部直接使用
    """
    print(f"\n=== 載入 {input_path} ===")
    with open(input_path, "r", encoding="utf-8") as f:
        clusters = json.load(f)
    print(f"讀取到 {len(clusters)} 個 cluster")
    for c in clusters:
        print(f"  {c['cluster_id']}：{c['total_count']} 筆事件，"
              f"representative_logs {len(c['representative_logs'])} 筆")
    return clusters

4-2：為 representative_logs 加上 event_id

In [31]:
def assign_event_ids(cluster: dict) -> list:
    logs = cluster.get("representative_logs", [])
    enriched = []
    for log in logs:
        enriched.append({
            "log_id":    log.get("log_id"),
            "timestamp": log.get("timestamp", ""),
            "text":      log.get("text", ""),
        })
    return enriched

4-3：System Prompt

In [32]:
SYSTEM_PROMPT = """
You are a senior SOC analyst assistant.
Analyze the provided security event logs and generate a structured incident report.

Rules you MUST follow:
1. Every claim in timeline and mitre_mapping MUST cite at least one log_id (e.g., log_0001).
2. Use ONLY information present in the provided log texts. Do NOT invent IPs, users, or actions.
3. If information is insufficient, write "Insufficient evidence" instead of guessing.
4. MITRE technique IDs must follow the format TXXXX or TXXXX.XXX (e.g., T1078.001).
5. Return ONLY valid JSON matching the schema. No markdown outside the JSON.
6. Respond in Traditional Chinese.
7. indicators_of_compromise 的 users 欄位必須包含 involved_entities 裡的所有使用者。
"""

OUTPUT_SCHEMA = {
    "incident_title": "<簡短攻擊事件標題>",
    "risk_level": "<Critical | High | Medium | Low>",
    "cluster_id": "<cluster_id>",
    "time_range": {
        "start": "<開始時間>",
        "end": "<結束時間>"
    },
    "executive_summary": {
        "description": "<2-3 句摘要：攻擊者、目標、手法、時間範圍>",
        "evidence": ["<log_id>"]
    },
    "timeline": [
        {
            "time": "<timestamp>",
            "summary": "<這個時間點發生了什麼>",
            "mitre_tactic": "<Privilege Escalation | Defense Evasion | Reconnaissance | ...>",
            "mitre_technique": "<T1078.001>",
            "evidence": {
                "log_id": "<log_id>",
                "log_text_excerpt": "<引用 log text 中的關鍵語句>"
            }
        }
    ],
    "mitre_attack_mapping": [
        {
            "technique_id": "<T1078.001>",
            "technique_name": "<Valid Accounts: Local Accounts>",
            "tactic": "<Privilege Escalation>",
            "observed_behavior": "<觀察到的具體行為>",
            "evidence": ["<log_id>"]
        }
    ],
    "attack_story": "<完整攻擊敘事，說明整個攻擊流程的前因後果>",
    "uncertainty": "<說明哪些資訊不足或無法確定的部分>",
    "recommended_triage_actions": ["<具體處置建議>"],
    "indicators_of_compromise": {
        "ips": ["<可疑 IP>"],
        "users": ["<可疑帳號>"],
        "windows_event_ids": ["<Windows Event ID，如 4688>"]
    }
}

4-4：呼叫 Gemini 生成報告

In [33]:
MODEL_NAME = "gemini-2.5-flash"
RETRY_COUNT = 3
RETRY_DELAY = 90      # 429 時等 90 秒
CLUSTER_DELAY = 30    # 每個 cluster 之間等 30 秒

def generate_report(cluster: dict, events: list) -> dict:
    cluster_id = cluster.get("cluster_id", "unknown")

    llm_input = {
        "task": "Generate a structured incident report from the following security event cluster.",
        "constraints": [
            "Cite log_id for every claim.",
            "Use only information from the provided log texts.",
            "Do not invent data.",
            "Return JSON only."
        ],
        "cluster_id": cluster_id,
        "time_range": cluster.get("time_range", {}),
        "involved_entities": cluster.get("involved_entities", {}),
        "representative_logs": events
    }

    prompt = (
        f"{json.dumps(llm_input, ensure_ascii=False)}\n\n"
        f"Output schema:\n{json.dumps(OUTPUT_SCHEMA, indent=2, ensure_ascii=False)}"
    )

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL_NAME}:generateContent"
    params = {"key": os.environ.get("GEMINI_API_KEY")}
    body = {
        "system_instruction": {"parts": [{"text": SYSTEM_PROMPT}]},
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {
            "temperature": 0.2,
            "response_mime_type": "application/json"
        }
    }

    for attempt in range(RETRY_COUNT):
        try:
            resp = requests.post(url, params=params, json=body, timeout=300)

            if resp.status_code == 429:
                wait = RETRY_DELAY * (attempt + 1)
                print(f"  [429] 配額限制，等待 {wait} 秒後重試（第 {attempt+1}/{RETRY_COUNT} 次）...")
                time.sleep(wait)
                continue

            if resp.status_code == 503:
                print(f"  [503] 伺服器忙碌，等待 30 秒後重試（第 {attempt+1}/{RETRY_COUNT} 次）...")
                time.sleep(30)
                continue

            resp.raise_for_status()
            text = resp.json()["candidates"][0]["content"]["parts"][0]["text"]
            report = json.loads(text)
            report["cluster_id"] = cluster_id
            print(f"  JSON 解析成功")
            return report

        except requests.exceptions.Timeout:
            print(f"  [TIMEOUT] 第 {attempt+1}/{RETRY_COUNT} 次逾時，重試中...")
            time.sleep(60)
            continue

        except Exception as e:
            print(f"  [ERROR] {e}")
            if hasattr(e, 'response') and e.response is not None:
                print(f"  [DETAIL] {e.response.text}")
            if attempt == RETRY_COUNT - 1:
                return {"error": str(e), "cluster_id": cluster_id}
            time.sleep(30)
            continue

    return {"error": "max retries exceeded", "cluster_id": cluster_id}

4-5：Evidence Grounding 驗證

4-6：存檔

In [34]:
def save_report(report: dict, output_dir: str) -> str:
    """
    將報告存成 JSON 檔
    檔名格式：report_cluster_000.json
    """
    os.makedirs(output_dir, exist_ok=True)
    cluster_id = report.get("cluster_id", "unknown")
    path = os.path.join(output_dir, f"report_{cluster_id}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    return path

4-7：印出摘要

In [35]:
def print_summary(report: dict):
    if "error" in report:
        print(f"  [ERROR] {report['error']}")
        return
    print(f"  標題：    {report.get('incident_title', 'N/A')}")
    print(f"  風險等級：{report.get('risk_level', 'N/A')}")
    print(f"  Timeline：{len(report.get('timeline', []))} 個步驟")
    print(f"  MITRE：   {len(report.get('mitre_attack_mapping', []))} 個 technique")


print("所有函式定義完成")

所有函式定義完成


# Cell 5：執行完整 Pipeline
Step 1：載入 Phase 2 輸出

In [36]:
# Step 1：載入 Phase 2 輸出
clusters = load_clusters(INPUT_PATH)

if not clusters:
    print("\n[WARNING] 沒有有效 cluster，請確認 clustered_events.json 是否正確上傳。")
else:
    print(f"\n=== 開始處理 {len(clusters)} 個 cluster ===")

    all_reports = []

    for cluster in clusters:
        cluster_id = cluster.get("cluster_id")
        print(f"\n--- 處理 {cluster_id} ---")

        # Step 2：對齊 log_id
        events = assign_event_ids(cluster)
        print(f"  representative_logs：{len(events)} 筆，"
              f"log_id {events[0]['log_id']} ~ {events[-1]['log_id']}")

        # Step 3：呼叫 Gemini 生成報告
        print(f"  呼叫 {MODEL_NAME}...")
        report = generate_report(cluster, events)

        # Step 4：存檔
        path = save_report(report, OUTPUT_DIR)
        print(f"  已儲存：{path}")

        # Step 5：印出摘要
        print_summary(report)

        all_reports.append(report)

        # 每個 cluster 處理完之後等待，避免短時間打太多次 API
        if cluster != clusters[-1]:  # 最後一個不用等
            print(f"  等待 {CLUSTER_DELAY} 秒再處理下一個 cluster...")
            time.sleep(CLUSTER_DELAY)

    print(f"\n=== 全部完成，{len(all_reports)} 份報告已存至 {OUTPUT_DIR}/ ===")



=== 載入 ./clustered_events.json ===
讀取到 6 個 cluster
  cluster_000：5311 筆事件，representative_logs 30 筆
  cluster_001：256 筆事件，representative_logs 30 筆
  cluster_002：55 筆事件，representative_logs 30 筆
  cluster_003：112 筆事件，representative_logs 30 筆
  cluster_004：12 筆事件，representative_logs 12 筆
  cluster_005：20 筆事件，representative_logs 20 筆

=== 開始處理 6 個 cluster ===

--- 處理 cluster_000 ---
  representative_logs：30 筆，log_id E007018 ~ E004218
  呼叫 gemini-2.5-flash...
  JSON 解析成功
  已儲存：phase3_output_07/report_cluster_000.json
  標題：    可疑的DNS連線嘗試與網路探測
  風險等級：Medium
  Timeline：30 個步驟
  MITRE：   2 個 technique
  等待 30 秒再處理下一個 cluster...

--- 處理 cluster_001 ---
  representative_logs：30 筆，log_id E000513 ~ E000542
  呼叫 gemini-2.5-flash...
  [503] 伺服器忙碌，等待 30 秒後重試（第 1/3 次）...
  JSON 解析成功
  已儲存：phase3_output_07/report_cluster_001.json
  標題：    可疑網路連線嘗試
  風險等級：Medium
  Timeline：1 個步驟
  MITRE：   1 個 technique
  等待 30 秒再處理下一個 cluster...

--- 處理 cluster_002 ---
  representative_logs：30 筆，log_id E002494 ~ E002399

# Cell 6：查看單份報告內容

In [44]:
# 修改 index 查看不同的報告
# 0 = 第一份，1 = 第二份

REPORT_INDEX = 6

if all_reports and REPORT_INDEX < len(all_reports):
    report = all_reports[REPORT_INDEX]
    print(json.dumps(report, indent=2, ensure_ascii=False))
else:
    print("沒有報告可顯示，請先執行 Cell 5")


沒有報告可顯示，請先執行 Cell 5


In [38]:
# 輸出合併給 Evaluation 模組使用

all_reports_path = os.path.join(OUTPUT_DIR, "all_reports.json")
with open(all_reports_path, "w", encoding="utf-8") as f:
    json.dump(all_reports, f, indent=2, ensure_ascii=False)
print(f"合併報告已存至：{all_reports_path}")


合併報告已存至：phase3_output_07/all_reports.json
